<a href="https://colab.research.google.com/github/saffarizadeh/INSY5378/blob/main/Text_Classification_and_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://kambizsaffari.com/Logo/College_of_Business.cmyk-hz-lg.png" width="500px"/>

# *INSY 5378 - Advanced AI*

# **Text Classification, Language Models, and the Transformer**

Instructor: Dr. Kambiz Saffari

---

Note: You MUST read the chapters. Going through this notebook does not replace the value of reading the chapters.

This notebook covers two chapters together because they tell one continuous story: how to feed raw text into a neural network, and how the Transformer architecture (which powers modern systems like ChatGPT) actually works.

- Chapter 14, Text classification: https://deeplearningwithpython.io/chapters/chapter14_text-classification/
- Chapter 15, Language models and the Transformer: https://deeplearningwithpython.io/chapters/chapter15_language-models-and-the-transformer/

> *Disclaimer: This notebook is a personal study guide created for educational purposes. It summarizes and references material from "Deep Learning with Python, Third Edition" by François Chollet and Matthew Watson (Manning Publications, 2025). All conceptual content and code examples are derived from the book and the companion notebooks. This material is shared strictly for non-commercial classroom use under fair use; please support the authors by purchasing the book.*

## Setup

### Use a GPU runtime

Several models in this notebook (especially the LSTM, the Transformer translator, and the RoBERTa fine-tuning section) will be painfully slow on a CPU. **Before running any code below, switch your Colab runtime to a GPU**:

> `Runtime` > `Change runtime type` > `Hardware accelerator: T4 GPU` (or any available GPU) > `Save`

### Install and configure Keras

We use Keras 3 with the JAX backend throughout. We also install `keras-hub`, which gives us pretrained Transformer models like RoBERTa later in the notebook.

In [ ]:
!pip install keras keras-hub --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-nlp 0.26.0 requires keras-hub==0.26.0, but you have keras-hub 0.27.1 which is incompatible.


In [ ]:


import os
os.environ["KERAS_BACKEND"] = "jax"

# Part I: Text Classification (Chapter 14)

In Chapter 4, we already trained a sentiment classifier on the IMDb movie review dataset. But Keras handed us a version of the data that had **already** been turned into integer sequences. In real applications, you start with raw text files, and you have to design every preprocessing step yourself. That is what this chapter is about.

By the end of Part I you will be able to:

1. Convert raw text into integer sequences using character, word, and **subword** tokenizers.
2. Decide whether to treat text as an unordered **set** of tokens (bag-of-words) or as an ordered **sequence**.
3. Train both kinds of models on raw IMDb data and compare them.
4. Understand why **word embeddings** are dramatically more efficient than one-hot encoding for sequence models.

## A brief history of NLP

Natural language processing (NLP) is the field of getting computers to read, write, and manipulate human language. For decades, NLP was dominated by handcrafted grammatical rules: linguists would write down explicit if-then rules for how English (or any other language) works, and a program would apply them. That approach plateaued in the 1990s, and statistical machine learning approaches gradually took over. In the 2010s, recurrent neural networks (especially LSTMs) gave the field a big jump in accuracy. Then in 2017, the **Transformer** architecture changed everything, and is what makes modern systems like ChatGPT possible. We will get to it in Part II of this notebook.

The unifying view across all of these eras is simple: **NLP is pattern recognition applied to words in text**. Spam filtering, machine translation, next-word prediction, and chat responses are all instances of finding statistical regularities in sequences of tokens.

## Preparing text data

Neural networks consume tensors of numbers, not strings. So before any modeling can happen, we need a pipeline that turns a sentence like

> *"The quick brown fox jumped over the lazy dog."*

into a sequence of integers that a model can actually process. That pipeline almost always has three stages:

1. **Standardization.** Light text-to-text cleanup, like lowercasing everything and stripping accent marks. The goal is to remove differences that you do not want the model to worry about (so that "Mexico" and "México" are treated as the same word, for example).
2. **Splitting.** Chopping the text into small units called *tokens*. A token might be a character, a word, or a piece of a word.
3. **Indexing.** Looking each token up in a fixed table called the *vocabulary*, which assigns every known token a unique integer ID.

The whole three-step process is called **tokenization**, and a software object that performs it is called a **tokenizer**.

![Text preprocessing pipeline](https://deeplearningwithpython.io/images/ch14/text-pipeline.c09bbad6.png)

### What if the tokenizer sees a word it has never seen before?

The vocabulary is built from training data, so it only knows about words that showed up there. If we later try to tokenize a sentence containing a word the vocabulary has never seen (say, a rare proper noun, a slang term, or a typo), the lookup will fail.

To avoid that, every vocabulary reserves one special slot for an **unknown token**, conventionally written as `[UNK]`. Whenever the tokenizer encounters a word that is not in its vocabulary, it maps that word to `[UNK]` (which always has a fixed integer ID, usually 0). This way, the tokenizer never crashes on unfamiliar input. The model just sees an `[UNK]` placeholder where the unknown word would have been. Later in this notebook you will see another reserved token, `[PAD]`, which serves a different purpose (filling shorter sentences out to a fixed length).

### Three flavors of tokenizer

There are three main families of tokenizers, and the choice between them is one of the most important decisions when working with text:

1. **Character-level.** Every individual character (letter, digit, punctuation mark) becomes a token. The vocabulary is tiny, often just a few dozen entries. But each input becomes a very long sequence of tokens, which is slow for the model to process.
2. **Word-level.** Every whitespace-separated word becomes a token. Sequences are short, but the vocabulary explodes (English has hundreds of thousands of words), and any word the vocabulary cannot fit becomes `[UNK]`, which throws away information.
3. **Subword.** A clever hybrid. Common words like "the" stay as single tokens, while rare or unusual words get broken into pieces. For example, the rare word "foxhound" might get split into "fox" and "hound". This is what virtually every modern model (GPT, BERT, RoBERTa, and so on) uses.

Character and word tokenizers are simple enough to implement in a few lines of regex plus a dictionary lookup, so we will not dwell on them. The interesting one is **subword tokenization**, because the algorithm behind it (called **Byte-Pair Encoding**, or BPE) is genuinely clever, and it is the algorithm that powers ChatGPT's tokenizer to this day. Let's build it from scratch on a tiny example.

## Byte-Pair Encoding (BPE), the algorithm behind subword tokenizers

Almost every modern tokenizer (the ones used by GPT, BERT, RoBERTa, Llama, and so on) is built with an algorithm called **Byte-Pair Encoding**, or BPE for short. We will not implement it from scratch, but you should understand the idea, because later in the notebook we will load RoBERTa's pretrained BPE tokenizer and treat it as a black box.

The idea is simple to state. Start with a vocabulary of single characters, then repeatedly merge the most frequent adjacent pair of symbols into a new combined symbol, until the vocabulary reaches the desired size.

In pseudocode:

```
vocabulary = {all individual characters in the training corpus}
while len(vocabulary) < desired_size:
    pair = most frequent adjacent symbol pair in the corpus
    merge every occurrence of that pair into a single new symbol
    add the new symbol to the vocabulary
```

To make this concrete, imagine we run BPE on a tiny training corpus consisting of just three sentences:

```
the quick brown fox
the slow brown fox
the quick brown foxhound
```

The first merge would be `("o", "w")`, because `ow` is the most frequent adjacent pair (it appears in `brown` three times and in `slow` once). After that merge, both `brown` and `slow` contain `ow` as a single symbol. The next merge would combine `t` and `h` into `th`. The merge after that combines `th` and `e` into `the`. And so on. After enough merges, frequent whole words like `the`, `brown`, and `fox` exist as single tokens in the vocabulary, while a rare word like `foxhound` still ends up split into a few subword pieces (perhaps `fox` plus `h`, `o`, `u`, `n`, `d`).

This is exactly the behavior we wanted. Common words get short single-token representations, rare words still get a representation made of pieces, and we never need an `[UNK]` token (in the worst case any unfamiliar word can always be broken down into individual characters, which are guaranteed to be in the vocabulary).

The key fact to remember when we use a pretrained BPE tokenizer later: **a BPE tokenizer is fully described by (a) the list of token symbols and (b) the ordered list of merge rules used to build them.** To tokenize a new piece of text, you split it into characters and re-apply the same merges in the same order. If you want to see a from-scratch implementation, the book walks through one in Chapter 14.

## Sets vs. sequences

Once we know how to turn text into integers, the next big design choice is: **how do we handle the order of those integers?** There are two camps.

- **Set models** ignore order entirely. A movie review is treated as just a "bag" of words: the model only knows which words are present, not in what order they appeared. They are simple, train in seconds, and (as we will note in a moment) are surprisingly competitive on tasks like sentiment classification. We will not build one in this notebook, but we will reference what they achieve.
- **Sequence models** consume the integer sequence in order: one token at a time (recurrent networks like LSTMs), or all at once with positional information (Transformers). They are much more powerful in principle, but they are also much hungrier for data. These are the models we actually build and the focus of the rest of the notebook.

First we need to load the IMDb data.

## Loading the raw IMDb dataset

In Chapter 4, we used the pre-tokenized version of IMDb that ships with Keras. Here we will work with the **raw text files**, just like you would in a real project. The dataset is a tarball of 50,000 movie reviews split into a train half and a test half, with each half further split into positive (`pos/`) and negative (`neg/`) subdirectories. We download it, set aside 20% of the training files as a validation set, and load all three splits with Keras's `text_dataset_from_directory` utility, which builds a `tf.data.Dataset` from a folder of labeled text files.

In [ ]:
import os, pathlib, shutil, random
import keras
from keras.utils import text_dataset_from_directory

# Download and extract the IMDb tarball.
zip_path = keras.utils.get_file(
    origin="https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz",
    fname="imdb",
    extract=True,
)
imdb_extract_dir = pathlib.Path(zip_path) / "aclImdb"

# Boilerplate: copy files into train/val/test directories so we can use
# text_dataset_from_directory. Nothing here is conceptually important;
# the only thing to know is that 20% of the training set becomes the
# validation set.
train_dir = pathlib.Path("imdb_train")
val_dir   = pathlib.Path("imdb_val")
test_dir  = pathlib.Path("imdb_test")
shutil.copytree(imdb_extract_dir / "test", test_dir, dirs_exist_ok=True)
for category in ("neg", "pos"):
    src_dir = imdb_extract_dir / "train" / category
    src_files = os.listdir(src_dir)
    random.Random(1337).shuffle(src_files)
    num_val = int(len(src_files) * 0.2)
    os.makedirs(val_dir / category, exist_ok=True)
    os.makedirs(train_dir / category, exist_ok=True)
    for f in src_files[:num_val]:
        shutil.copy(src_dir / f, val_dir / category / f)
    for f in src_files[num_val:]:
        shutil.copy(src_dir / f, train_dir / category / f)

# Load the three splits as tf.data.Datasets of (text, label) pairs.
batch_size = 32
train_ds = text_dataset_from_directory(train_dir, batch_size=batch_size)
val_ds   = text_dataset_from_directory(val_dir,   batch_size=batch_size)
test_ds  = text_dataset_from_directory(test_dir,  batch_size=batch_size)

84125825/84125825 ━━━━━━━━━━━━━━━━━━━━ 32s 0us/step
Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


We end up with 20,000 training reviews, 5,000 validation reviews, and 25,000 test reviews. Each batch is a tuple of `(strings, labels)`, where `strings` is a batch of raw byte strings (the actual text of each review) and `labels` is a batch of 0/1 integers (negative or positive). Keras has not done any tokenization for us. That is our job, starting now.

## A note on set models (and our baseline number)

Before we dive into sequence models, it is worth knowing what the simpler approach can do, because we will use it as a reference point throughout the rest of the notebook.

The simplest reasonable thing you can do with text is treat each review as an unordered **multi-hot vector**: a fixed-size binary vector with one dimension per vocabulary word, where dimension *i* is 1 if word *i* appears anywhere in the review and 0 otherwise. Then you train a single `Dense` layer with a sigmoid on top. That is just logistic regression over word presence, and it has roughly one weight per vocabulary word.

The book trains two versions of this on the IMDb data:

- A **unigram bag-of-words** model using a 20K-word vocabulary. About 20K parameters. Reaches about **88% test accuracy**.
- A **bigram model** that adds the most frequent two-word phrases (like `"not bad"` and `"one of"`) to the vocabulary alongside single words. About 30K parameters. Reaches about **90% test accuracy**.

Both train in under a minute on a CPU. Keras has a `TextVectorization` layer that handles the tokenization, vocabulary learning, and multi-hot encoding in a single call (with `output_mode="multi_hot"` and optionally `ngrams=2`); see Chapter 14 if you want the implementation details.

Why mention this at all if we are not going to build it? Two reasons:

1. **It is the bar that everything else in this notebook has to clear.** The 2-million-parameter LSTM we build next gets *worse* accuracy than the 30K-parameter bigram model. That surprising result is the entire motivation for pretraining, and pretraining is the entire motivation for the Transformer story in Part II.
2. **It is a useful reality check in your own work.** When you build a fancy sequence model and it underperforms a five-line bag-of-words baseline, you have learned something important about your dataset (it is probably too small) before you spend a week tuning the fancy model.

Now let's set up the few things we will actually need for the sequence models that follow.

In [ ]:
from keras import layers

# We will reuse this dataset of just the texts (no labels) to adapt our
# TextVectorization layer in the next section.
train_ds_no_labels = train_ds.map(lambda x, y: x)

# We will also reuse this early-stopping callback for the LSTM.
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    restore_best_weights=True,
    patience=2,
)

## Sequence models

Sequence models look at the integer sequence of token IDs in order. To build one, we need to handle one new wrinkle: reviews have different lengths, but a GPU expects every input in a batch to have the same shape. We solve this with **padding**.

The idea is to pick a fixed maximum sequence length (we will use 600 tokens, which covers about 95% of IMDb reviews in full), then for each review either truncate it (if longer) or pad it with a special `[PAD]` token (if shorter). The `[PAD]` token is always assigned ID 0, so the input becomes something like:

```
[12, 5, 88, 17, 6, 4, 0, 0, 0, 0]
```

The model then has to learn to ignore the trailing zeros. We will see how to do that cleanly using "masking" in a moment. Once again, `TextVectorization` handles all of this for us.

In [ ]:
max_length = 600
max_tokens = 30_000
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    split="whitespace",
    output_mode="int",                       # output integer IDs, not multi-hot
    output_sequence_length=max_length,       # pad / truncate to 600 tokens
)
text_vectorization.adapt(train_ds_no_labels)

sequence_train_ds = train_ds.map(lambda x, y: (text_vectorization(x), y), num_parallel_calls=8)
sequence_val_ds   = val_ds.map(lambda x, y: (text_vectorization(x), y), num_parallel_calls=8)
sequence_test_ds  = test_ds.map(lambda x, y: (text_vectorization(x), y), num_parallel_calls=8)

In [ ]:
x, y = next(sequence_test_ds.as_numpy_iterator())
print("Shape:", x.shape)
print("Example sequence (notice the trailing zeros are padding):")
print(x[0])

Shape: (32, 600)
Example sequence (notice the trailing zeros are padding):
[   10   403   255    11    14     2   169   172    15     1  8718    13
    11    20    44     9    32     4   368   113  4330  2018     4  2859
    12    44   159     6    79    16     2   277   113    49    38   122
     3   476  2435   102     9    14  1325     6   104 11767  1325     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0 

Each batch now has shape `(32, 600)`: 32 reviews, each represented as a sequence of 600 integer token IDs, with trailing zeros where the original review was shorter than 600 tokens.

### Aside: do we actually need to cap the sequence length?

You may be wondering why we are capping the input length at all. We have told you before that an RNN can in principle handle a sequence of any length, and that is still true. An LSTM cell is just a fixed function that takes `(input_token, previous_state)` and returns `(output, new_state)`. The same weights are reused at every time step, so the cell itself does not care how many time steps you call it on. The cap is purely a **batching** concern, not a model-architecture concern.

Two practical reasons force us to cap the length **at training time**:

1. **Batches must be rectangular.** When we stack 32 reviews into one tensor for the GPU, every row has to be the same length, so the longest review in the batch determines the cost for everyone in that batch. Without a cap, one freakishly long 3,000-token review would force every other review in its batch to be padded to 3,000 tokens, blowing up memory and wasting compute on positions that are mostly padding.
2. **Memory scales linearly with sequence length during training.** Backpropagation through time has to store every intermediate hidden state for the backward pass, so a 3,000-token review costs roughly 5x the activation memory of a 600-token one. The hard wall on a GPU is usually memory, not speed.

Capping at 600 (which covers about 95% of IMDb reviews in full) puts a ceiling on the worst case so we can use a reasonable batch size and finish training in a reasonable time.

**At inference time, almost all of those reasons go away.** Inference is a single forward pass with no stored activations for backprop, so memory usage is a small fraction of what training requires. You typically process one example (or a small batch of similar-length examples) at a time, so the "longest member drags the whole batch" problem largely disappears. And if a user hands your model a 2,500-word review, truncating it is throwing away signal the user expected you to use.

So at inference time you really *can* feed an RNN-based model arbitrary-length input, with two soft caveats:

1. **Generalization past training lengths is not guaranteed.** A model trained only on sequences up to 600 tokens may behave oddly on a 5,000-token input, simply because that length is out-of-distribution for the model. This is usually a soft degradation, not a cliff.
2. **RNNs forget over long distances.** Feeding in 5,000 tokens does not mean the model is *using* all 5,000; the early ones will have faded from the hidden state by the time the loop reaches the end. (This is one of the limitations the Transformer was designed to fix, as we will see in Part II.)

#### How to actually do this in Keras

There is one gotcha. In a moment we will write something like:

```python
inputs = keras.Input(shape=(max_length,), dtype="int32")
```

That bakes the fixed length 600 into the model's input signature, and Keras will reject inputs of any other length even at inference time. The fix is to declare the sequence dimension as variable by passing `None` instead:

```python
inputs = keras.Input(shape=(None,), dtype="int32")
```

With `shape=(None,)`, the same trained weights will accept inputs of any length at inference time. Nothing inside the LSTM cell cares.

If your model already exists with a fixed length and you do not want to retrain, you can build a "twin" model with `shape=(None,)` and the same layers, then copy the trained weights over with `new_model.set_weights(old_model.get_weights())`. (We use this exact trick for a different reason in the Shakespeare generation section: same layers, copied weights, but a structurally different inference interface.)

One important caveat: every layer in the stack has to tolerate variable length. RNN layers, attention layers, 1D convolutions, embeddings, and layer norm all do. The most common offender that does *not* is `Flatten()` followed by `Dense()`, because the `Dense` needs a known number of input features. If your architecture has that pattern, swap the `Flatten` for a length-agnostic pooling layer like `GlobalAveragePooling1D()` first.

The cleanest habit is to **always declare sequence inputs with `shape=(None,)` from the start**, even during training. Then the same model handles both training (with batches padded to a fixed length per batch) and inference (with whatever length the user gives you), and you never need the twin-model trick.

For the rest of this section we will stick with `shape=(max_length,)` for simplicity, since it matches what the book does, but keep this aside in mind for your own projects.

### How do we feed token IDs to a neural network?

A neural network layer like `LSTM` or `Dense` expects floating-point vectors, not integer IDs. So we need to convert each token ID into a vector somehow. There are two options.

**Option 1: One-hot encoding.** Token ID 17 becomes a 30,000-dimensional vector with a 1 in position 17 and zeros everywhere else. This is the naive approach. It has two big problems:

1. The input tensor for a single batch becomes `(32, 600, 30_000)`, which is over 18 million floating-point numbers per batch. That is slow and wastes memory.
2. **It treats every word as completely unrelated to every other word.** Mathematically, all one-hot vectors are mutually orthogonal, which is the same as saying they have no relationship. So the model "thinks" that `"movie"` and `"film"` are as different from each other as `"movie"` and `"banana"`.

**Option 2: A learned word embedding.** Instead of a giant sparse one-hot vector, we represent each token as a small, dense vector of (say) 64 floating-point numbers. The values in these vectors are **learned during training**, just like any other model weights. During training, the vectors for related words get pushed close together in the embedding space, while unrelated words drift apart. Specific directions in the space end up encoding meaningful semantic relationships, like "from singular to plural" or "from a country to its capital."

![One-hot vs embedding](https://deeplearningwithpython.io/images/ch14/word-representations.b71fcc82.png)

A toy 2D illustration of what an embedding space might look like:

![Toy 2D word embedding space](https://deeplearningwithpython.io/images/ch14/word-embeddings.1bc937b3.png)

Mechanically, an `Embedding` layer is just a giant lookup table. You hand it an integer ID, and it returns the learned vector at that row of the table. The weights of the layer are the rows of the table.

![Embedding as a lookup table](https://deeplearningwithpython.io/images/ch14/embedding-dictionary.80faa429.png)

Word embeddings are a much better choice for sequence models than one-hot encoding. We will use them from now on.

### Training an LSTM with a word embedding

Now we have all the pieces we need to build a sequence model: an `Embedding` layer to turn token IDs into dense vectors, an LSTM to process the sequence (we covered LSTMs in detail in our earlier time-series notebook, so we will not re-derive them here), and a `Dense` layer at the end to make a binary prediction.

A few notes on the architecture:

- We use a **bidirectional** LSTM, which runs one LSTM forward over the sequence and another backward, then concatenates their outputs. This lets each position see context from both directions, which helps for classification (where we get to see the whole input before deciding).
- We pass `mask_zero=True` to the `Embedding` layer. This tells Keras: "token ID 0 means padding, please ignore those positions in everything downstream." The LSTM will use this mask automatically to skip padding positions.
- We add `Dropout` for regularization.

In [ ]:
hidden_dim = 64
inputs = keras.Input(shape=(max_length,), dtype="int32")
x = layers.Embedding(input_dim=max_tokens, output_dim=hidden_dim, mask_zero=True)(inputs)
x = layers.Bidirectional(layers.LSTM(hidden_dim))(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs, name="lstm_with_embedding")
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary(line_length=80)

Model: "lstm_with_embedding"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ input_layer           │ (None, 600)       │           0 │ -                  │
│ (InputLayer)          │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embedding (Embedding) │ (None, 600, 64)   │   1,920,000 │ input_layer[0][0]  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal (NotEqual)  │ (None, 600)       │           0 │ input_layer[0][0]  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ bidirectional         │ (None, 128)       │      66,048 │ embedding[0][0],   │
│ (Bidirectional)       │                   │             │ not_equal[0][0]    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout (Dropout)     │ (None, 128)       │           0 │ bidirectional[0][… │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dense (Dense)         │ (None, 1)         │         129 │ dropout[0][0]      │
└───────────────────────┴───────────────────┴─────────────┴────────────────────┘

 Total params: 1,986,177 (7.58 MB)

 Trainable params: 1,986,177 (7.58 MB)

 Non-trainable params: 0 (0.00 B)

**Where do these parameter counts come from?**

- **Embedding:** just a lookup table with one row per vocabulary word, so `input_dim * output_dim = 30000 * 64 = 1,920,000`.
- **Bidirectional LSTM:** recall from our from-scratch LSTM example that the layer has **four gates** (forget, input, candidate, output), and each gate owns three weight blocks: a `W` of shape `(H, I)`, a `U` of shape `(H, H)`, and a bias `b` of shape `(H,)`, where `I` is the input feature size and `H` is the hidden size. That gives `4 * H * (I + H + 1)` parameters per direction. Here `I = 64` (the embedding output) and `H = 64` (the LSTM units), so one direction has `4 * 64 * (64 + 64 + 1) = 33,024` parameters. **Bidirectional doubles it** to `66,048`. (The two `64`s are unrelated knobs that we just happened to set to the same value.)
- **Dense(1):** the bidirectional LSTM concatenates its forward and backward hidden states into a 128-dim vector, so this layer has `128 + 1 = 129` parameters.

In [ ]:
model.fit(
    sequence_train_ds,
    validation_data=sequence_val_ds,
    epochs=10,
    callbacks=[early_stopping],
)
test_loss, test_acc = model.evaluate(sequence_test_ds)
test_acc

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 55s 81ms/step - accuracy: 0.7985 - loss: 0.4331 - val_accuracy: 0.8740 - val_loss: 0.3096
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 51s 82ms/step - accuracy: 0.9230 - loss: 0.2069 - val_accuracy: 0.8894 - val_loss: 0.2968
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 49s 79ms/step - accuracy: 0.9599 - loss: 0.1125 - val_accuracy: 0.8856 - val_loss: 0.3748
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 49s 78ms/step - accuracy: 0.9808 - loss: 0.0594 - val_accuracy: 0.8772 - val_loss: 0.4052
782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 16ms/step - accuracy: 0.8682 - loss: 0.3454


0.8681599497795105

You will likely see something around **84% test accuracy**. That is *worse* than the 90% bigram baseline we mentioned earlier. What is going on?

The answer is **data**. A sequence model with about 2 million parameters needs a lot more training data than 20,000 labeled reviews to simultaneously learn (a) good word embeddings and (b) a good classifier on top of them. The set-based bigram model only had 30,001 parameters, so it generalizes happily on this dataset. The sequence model has so much capacity that it overfits before it can fully learn the structure of language.

This is the **data-hunger problem**, and it is the most important practical problem in modern NLP. There are two solutions to it:

1. **Pretrain the embeddings separately on a huge unlabeled corpus** (as in the Word2Vec or CBOW techniques). The book demonstrates this on the IMDb unsupervised data and gets the LSTM up to roughly bag-of-words level. The general idea is summarized in this figure, where the model learns to predict a word from its surrounding context:

![CBOW: predict the middle word from its context](https://deeplearningwithpython.io/images/ch14/cbow.01aaf529.png)

2. **Pretrain an entire Transformer model on billions of words from the web, then fine-tune it on your specific task.** This is the modern industrial approach, and it is what Part II of this notebook builds toward.

### Takeaway from Chapter 14

| Model | Test accuracy | Parameters | Training time |
|---|---|---|---|
| Bag-of-words (unigrams) | ~88% | 20K | seconds |
| Bag-of-words (bigrams) | ~90% | 30K | seconds |
| LSTM with learned embedding | ~84% | 2M | minutes |

For *small* labeled datasets, simple set-based models are remarkably hard to beat. To make sequence models pay off, we need pretraining on lots of data, which is exactly where the Transformer comes in.

# Part II: Language Models and the Transformer (Chapter 15)

Text classification only requires the model to output a single number per review (the predicted sentiment). Many other text problems are harder: machine translation, question answering, summarization, and chat all require the model to output an entire sequence of words. For these we need a **language model**.

By the end of Part II you will be able to:

1. Train a small character-level language model and use it to generate Shakespeare-flavored text.
2. Build a sequence-to-sequence translation model that translates English to Spanish.
3. Understand the **attention** mechanism from first principles.
4. Build a full **Transformer** encoder-decoder from scratch.
5. Fine-tune a **pretrained Transformer** (RoBERTa) and beat all of Part I's results on IMDb.

## What is a language model?

A **language model** is a model that learns the probability of the next token given all previous tokens:

$$ p(\text{next token} \mid \text{past tokens}) $$

That sounds modest, but here is the trick: if you can predict one token at a time, you can generate **arbitrarily long text** by calling the model in a loop. Predict a token, append it to the input, predict the next, append it, and so on. A model used in this feedback loop is called **autoregressive**.

We will warm up with a tiny example: a character-level language model trained on a few of Shakespeare's plays. The model will not be very good (we are training a small model on a small corpus for a few minutes), but it will be enough to demonstrate the entire end-to-end recipe of training a language model and then sampling text from it.

### Preparing the Shakespeare data

We download a public-domain text file containing several of Shakespeare's plays, chop the text into chunks of 100 characters each, and pair every chunk with a label sequence that is the same chunk shifted forward by one character. So at every position in the input, the label tells the model which character came next in the original text. We then build a character-level vocabulary with `TextVectorization` (just 67 unique characters cover the entire corpus) and tokenize. None of this is conceptually about language modeling; it is plain data plumbing.

In [ ]:
import tensorflow as tf

# Download and load the raw text.
filename = keras.utils.get_file(
    origin="https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt",
)
shakespeare = open(filename, "r").read()

# Chop into 100-character chunks. Labels = chunks shifted by one character.
sequence_length = 100
features = [shakespeare[i : i + sequence_length]     for i in range(0, len(shakespeare) - 1, sequence_length)]
labels   = [shakespeare[i + 1 : i + 1 + sequence_length] for i in range(0, len(shakespeare) - 1, sequence_length)]
dataset = tf.data.Dataset.from_tensor_slices((features, labels))

# Learn a character-level vocabulary and tokenize.
tokenizer = layers.TextVectorization(
    standardize=None,
    split="character",
    output_sequence_length=sequence_length,
)
tokenizer.adapt(dataset.map(lambda text, labels: text))
vocabulary_size = tokenizer.vocabulary_size()

dataset = dataset.map(lambda f, l: (tokenizer(f), tokenizer(l)), num_parallel_calls=8)
training_data = dataset.shuffle(10_000).batch(64).cache()
print("Vocabulary size:", vocabulary_size)

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
Vocabulary size: 67


### Building the model

A single GRU layer is more than enough for this small problem. (A GRU is a simpler cousin of the LSTM. We covered both in detail in the time-series notebook.) Two important details:

- We pass `return_sequences=True` to the GRU. This means the GRU outputs a prediction at *every* position in the sequence, not just the final one. This is what makes this a language modeling setup: at every position, the model is asked "what character comes next here?", and we get one prediction per input character.
- The final `Dense` layer has `vocabulary_size` outputs and uses a softmax activation. Each output is the predicted probability that the next character will be character *i* in the vocabulary.

In [ ]:
embedding_dim = 256
hidden_dim = 1024

inputs = layers.Input(shape=(sequence_length,), dtype="int", name="token_ids")
x = layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x = layers.GRU(hidden_dim, return_sequences=True)(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(vocabulary_size, activation="softmax")(x)
model = keras.Model(inputs, outputs)
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)
model.fit(training_data, epochs=20)

Epoch 1/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 20s 92ms/step - loss: 2.7223 - sparse_categorical_accuracy: 0.2784
Epoch 2/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 14s 80ms/step - loss: 1.9945 - sparse_categorical_accuracy: 0.4160
Epoch 3/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 14s 77ms/step - loss: 1.7269 - sparse_categorical_accuracy: 0.4872
Epoch 4/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 75ms/step - loss: 1.5772 - sparse_categorical_accuracy: 0.5268
Epoch 5/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 74ms/step - loss: 1.4840 - sparse_categorical_accuracy: 0.5512
Epoch 6/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 74ms/step - loss: 1.4181 - sparse_categorical_accuracy: 0.5678
Epoch 7/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 75ms/step - loss: 1.3667 - sparse_categorical_accuracy: 0.5810
Epoch 8/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 76ms/step - loss: 1.3230 - sparse_categorical_accuracy: 0.5920
Epoch 9/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 13s 76ms/step - loss: 1.2824 - sparse_categorical_accuracy: 0.6029
Epoch 10/20
175/175 ━━━━━━━━━━━━━━━━━

In [ ]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ token_ids (InputLayer)          │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 100, 256)       │        17,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 100, 1024)      │     3,938,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 100, 1024)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100, 67)        │        68,675 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,072,395 (46.05 MB)

 Trainable params: 4,024,131 (15.35 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 8,048,264 (30.70 MB)

**Important note about training direction.** Notice that we used a plain (unidirectional) GRU, not a `Bidirectional(GRU(...))`. If we had used a bidirectional layer, the model would be able to peek at the future direction of the sequence and effectively see the very character it is supposed to predict. The training loss would crash to zero immediately, but generation would not work at all, because at generation time there is no "future" to peek at. Language models must be **causal**: information is only allowed to flow from past to future.

**Wait, how does this work? The labels are 100 characters long, but the final `Dense` layer only has 67 nodes.**

Both things are true, and the trick is that this model makes **100 predictions per training example**, not one. Trace the shapes through the model with a batch of 64:

```
inputs:                                  (64, 100)
after Embedding(67, 256):                (64, 100, 256)
after GRU(1024, return_sequences=True):  (64, 100, 1024)
after Dense(67, softmax):                (64, 100, 67)
```

The `Dense` layer in Keras only acts on the **last** axis of its input. When that input has shape `(64, 100, 1024)`, the same 67-node `Dense` layer (with the same weights, `1024 * 67 + 67 = 68,675` parameters total) is applied independently to each of the `64 * 100 = 6400` length-1024 vectors. The output is a softmax over 67 characters at every position.

The label tensor is built by shifting the input forward by one character, so its shape is `(64, 100)` and label position `i` is the integer ID of the character that actually came next at position `i`. `sparse_categorical_crossentropy` then computes one cross-entropy at each of the 6,400 positions and averages them. From a single 100-character chunk, the model gets 100 supervised next-character training signals out of one forward pass. That is the central efficiency trick of language modeling.

This is also why we used a plain (unidirectional) GRU, not a `Bidirectional(GRU(...))`. A unidirectional GRU at position `i` has only seen positions `0` through `i`, so its prediction of position `i+1` is genuinely a prediction. A bidirectional layer would let position `i` peek at position `i+1`, which is the very label we are trying to predict, and training would collapse to a trivial copy. Language models must be **causal**: information only flows from past to future.

### Generating text from the trained model

Now that we have a trained next-character predictor, we want to use it to actually generate text. The procedure is the autoregressive loop we described earlier: feed in a starting prompt, ask the model what character comes next, append that character to our running sequence, feed the new sequence back in, and repeat. In pseudocode:

```
generated = prompt
for step in range(num_chars_to_generate):
    next_char_probs = model(generated)[-1]   # distribution over the next character
    next_char = argmax(next_char_probs)       # pick the most likely one
    generated = generated + next_char
print(generated)
```

There are two practical wrinkles when implementing this in Keras with a recurrent model.

1. **Efficient state passing.** During training, the GRU's hidden state was managed implicitly inside each 100-character chunk. During generation, we want to feed *one* character at a time and explicitly carry the hidden state forward across calls, instead of re-running the model on the entire generated sequence after every new character. The standard trick is to build a "twin" of the trained model that takes a single token plus a state as inputs, returns a prediction plus the updated state, and shares weights with the trained model via `set_weights`.

2. **Priming the state with the prompt.** Before generation starts, we feed the prompt through the twin model character by character so the hidden state captures what the prompt "looks like." Only then do we start sampling new characters.

If you run the book's implementation, the model produces output that looks roughly like Shakespeare: character names followed by colons, line breaks between speakers, archaic vocabulary, and semi-coherent sentences. Not bad for a 4-million-parameter model trained for two minutes on a single text file.

This template (train a model that predicts one token from past tokens, then sample from it in a loop) is the foundation of every GPT-style language model in existence. Everything that follows in this chapter is about scaling this idea up and replacing the recurrent backbone with something better.

In [ ]:
import numpy as np

inputs = keras.Input(shape=(1,), dtype="int", name="token_ids")
input_state = keras.Input(shape=(hidden_dim,), name="state")

x = layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x, output_state = layers.GRU(hidden_dim, return_state=True)(
    x, initial_state=input_state
)
outputs = layers.Dense(vocabulary_size, activation="softmax")(x)
generation_model = keras.Model(
    inputs=(inputs, input_state), # the second input is the state you're threading through manually so the GRU doesn't forget between calls.
    outputs=(outputs, output_state),
)
generation_model.set_weights(model.get_weights())

tokens = tokenizer.get_vocabulary()
token_ids = range(vocabulary_size)
char_to_id = dict(zip(tokens, token_ids))
id_to_char = dict(zip(token_ids, tokens))

prompt = """
KING RICHARD III:
"""

input_ids = [char_to_id[c] for c in prompt]
state = keras.ops.zeros(shape=(1, hidden_dim))
for token_id in input_ids:
    inputs = keras.ops.expand_dims([token_id], axis=0)
    predictions, state = generation_model.predict((inputs, state), verbose=0)

generated_ids = []
max_length = 250
for i in range(max_length):
    next_char = int(np.argmax(predictions, axis=-1)[0])
    generated_ids.append(next_char)
    inputs = keras.ops.expand_dims([next_char], axis=0)
    predictions, state = generation_model.predict((inputs, state), verbose=0)

output = "".join([id_to_char[token_id] for token_id in generated_ids])
print(prompt + output)


KING RICHARD III:
What is the matter?

MENENIUS:
One that do content them.

ROMEO:
O thou wilt warrant him, betides to him the book of her
That will put forth the present at the heels.

GENRY BOLINGBROKE:
My Lord of Naples, father, for I know
What can you for your voi


## Sequence-to-sequence learning

Let's tackle a harder problem now. What if the input and output are *different* sequences? The classic example is **machine translation**: given an English sentence, produce its Spanish translation. This is called a **sequence-to-sequence** (or **seq2seq**) problem.

The general recipe uses two components:

- An **encoder** consumes the source sentence (in English) and produces an intermediate vector representation of it.
- A **decoder** is a language model (just like our Shakespeare model) that generates the target sentence (in Spanish) one token at a time. The crucial difference from our Shakespeare model is that the decoder also has access to the encoder's representation, so it knows what it is supposed to be translating.

During training we feed the decoder the *correct* previous target tokens (this is called **teacher forcing**, and it makes training stable). During inference we generate one token at a time and feed each predicted token back in.

![Sequence-to-sequence learning](https://deeplearningwithpython.io/images/ch15/seq2seq-learning.0e1e1c31.png)

We will build this in two flavors: first with RNNs (briefly, as a baseline), and then with Transformers (in detail).

### English-to-Spanish data

We use a public dataset of English-Spanish sentence pairs. Two things matter conceptually about how we set it up; everything else is plumbing.

1. **We wrap every Spanish sentence in `[start]` and `[end]` markers.** The decoder will use `[start]` as its very first input token, and it will learn to emit `[end]` when the translation is complete (so we know when to stop generating).

2. **We arrange every example as a teacher-forcing triple `(english, spanish_so_far, next_spanish_token)`.** Concretely, if the Spanish sentence is `[start] hola mundo [end]`, then at training time the decoder sees the English sentence plus the prefix `[start] hola mundo` and is asked to predict `hola mundo [end]` (the same prefix shifted one token forward). At every target position, it gets the correct previous tokens for free and only has to predict the *next* one. This is what makes seq2seq training stable. We also apply a sample-weight mask so that padding positions do not contribute to the loss.

The cell below downloads the data, builds two `TextVectorization` tokenizers (one for English, one for Spanish), splits into train/val/test, and packages everything as a `tf.data.Dataset`. The only fiddly bit is that we customize the Spanish standardization so that the `[` and `]` characters in our marker tokens survive (the default standardizer would strip them).

In [ ]:
import pathlib, string, re

# Download and parse the sentence pairs.
zip_path = keras.utils.get_file(
    origin="http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip",
    fname="spa-eng",
    extract=True,
)
text_path = pathlib.Path(zip_path) / "spa-eng" / "spa.txt"
with open(text_path) as f:
    lines = f.read().split("\n")[:-1]
text_pairs = []
for line in lines:
    english, spanish = line.split("\t")
    text_pairs.append((english, "[start] " + spanish + " [end]"))

# Train / val / test split.
random.shuffle(text_pairs)
val_samples = int(0.15 * len(text_pairs))
train_samples = len(text_pairs) - 2 * val_samples
train_pairs = text_pairs[:train_samples]
val_pairs   = text_pairs[train_samples : train_samples + val_samples]
test_pairs  = text_pairs[train_samples + val_samples :]

# Tokenizers. The Spanish standardizer strips punctuation but KEEPS [ and ]
# so the [start] / [end] marker tokens survive.
strip_chars = (string.punctuation + "¿").replace("[", "").replace("]", "")
def custom_standardization(s):
    return tf.strings.regex_replace(tf.strings.lower(s), f"[{re.escape(strip_chars)}]", "")

vocab_size = 15000
sequence_length = 20
english_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size, output_mode="int", output_sequence_length=sequence_length,
)
spanish_tokenizer = layers.TextVectorization(
    max_tokens=vocab_size, output_mode="int", output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)
english_tokenizer.adapt([p[0] for p in train_pairs])
spanish_tokenizer.adapt([p[1] for p in train_pairs])

# Build the teacher-forcing dataset: (english, spanish_so_far) -> next_spanish_token
batch_size = 64
def format_dataset(eng, spa):
    eng = english_tokenizer(eng)
    spa = spanish_tokenizer(spa)
    return ({"english": eng, "spanish": spa[:, :-1]}, spa[:, 1:], spa[:, 1:] != 0)

def make_dataset(pairs):
    eng_texts, spa_texts = zip(*pairs)
    ds = tf.data.Dataset.from_tensor_slices((list(eng_texts), list(spa_texts)))
    ds = ds.batch(batch_size).map(format_dataset, num_parallel_calls=4)
    return ds.shuffle(2048).cache()

train_ds = make_dataset(train_pairs)
val_ds   = make_dataset(val_pairs)

2638744/2638744 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step


### A quick RNN baseline

Before we get to the Transformer, let's see what a seq2seq RNN looks like. We will not dwell on it (we already covered RNNs in detail in our previous notebook on time-series); the goal here is just to have a baseline to compare the Transformer against.

The recipe: a **bidirectional GRU** reads the entire English sentence and produces a single summary vector. That vector becomes the **initial hidden state** of a second GRU (the decoder), which generates the Spanish translation one word at a time. A `Dense` softmax on top predicts the next Spanish word at each step.

![seq2seq with RNNs](https://deeplearningwithpython.io/images/ch15/seq2seq-rnn.ec377d3b.png)

In [ ]:
embed_dim = 256
hidden_dim = 1024

# --- Encoder: bidirectional GRU over the English sentence ---
source = keras.Input(shape=(None,), dtype="int32", name="english")
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(source)
encoder_output = layers.Bidirectional(layers.GRU(hidden_dim), merge_mode="sum")(x)

# --- Decoder: GRU initialized with the encoder output ---
target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(target)
x = layers.GRU(hidden_dim, return_sequences=True)(x, initial_state=encoder_output)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)

seq2seq_rnn = keras.Model([source, target], target_predictions)
seq2seq_rnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
seq2seq_rnn.fit(train_ds, epochs=15, validation_data=val_ds)

Epoch 1/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 131s 96ms/step - accuracy: 0.3570 - loss: 3.6735 - val_accuracy: 0.4932 - val_loss: 2.5082
Epoch 2/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 119s 91ms/step - accuracy: 0.5353 - loss: 2.2590 - val_accuracy: 0.5864 - val_loss: 1.9283
Epoch 3/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 119s 91ms/step - accuracy: 0.6207 - loss: 1.6347 - val_accuracy: 0.6228 - val_loss: 1.7349
Epoch 4/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 119s 91ms/step - accuracy: 0.6803 - loss: 1.2476 - val_accuracy: 0.6385 - val_loss: 1.6556
Epoch 5/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 119s 91ms/step - accuracy: 0.7272 - loss: 0.9905 - val_accuracy: 0.6447 - val_loss: 1.6579
Epoch 6/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 119s 91ms/step - accuracy: 0.7626 - loss: 0.8250 - val_accuracy: 0.6512 - val_loss: 1.6757
Epoch 7/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 119s 91ms/step - accuracy: 0.7860 - loss: 0.7223 - val_accuracy: 0.6491 - val_loss: 1.7058
Epoch 8/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 119s 91ms/step - accuracy: 

In [ ]:
import numpy as np

spa_vocab = spanish_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = english_tokenizer([input_sentence])
    decoded_sentence = "[start]"
    for i in range(sequence_length):
        tokenized_target_sentence = spanish_tokenizer([decoded_sentence])
        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = seq2seq_rnn.predict(inputs, verbose=0)
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

-
If the weather had been nice yesterday, we would have gone to the zoo.
[start] si el tiempo fue muy ocupado ayer me habría ido al final [end]
-
They're trying.
[start] están tratando [end]
-
We have to get up early tomorrow morning.
[start] tenemos que salir temprano para la mañana [end]
-
Their patience was about to give out.
[start] su paciencia estaba a punto de salir [end]
-
These are new.
[start] estas son nuevos [end]


You should reach roughly **65% next-token accuracy**. This works, but the architecture has two fundamental problems that motivate the entire Transformer:

1. **The encoder has to compress the entire source sentence into a single fixed-size vector.** Long sentences inevitably lose information, because no matter how long the source, it all has to fit into one 1024-dimensional hidden state.
2. **RNNs forget over long distances.** By the time the decoder is generating the 20th Spanish word, the GRU's internal state barely remembers the first English word it saw.

What if, instead of forcing all source information through this bottleneck, we let every word of the decoder **look directly at every word of the source** and decide which ones are relevant *right now*? That idea is called **attention**, and it changed everything.

## The Transformer architecture

In 2017, Vaswani and colleagues at Google published a paper called "Attention Is All You Need." Their key claim sounded outrageous at the time: you can throw away RNNs entirely and build a state-of-the-art sequence model out of attention layers alone. They called the result the **Transformer**, and within a few years it had taken over essentially all of NLP and a lot of computer vision and audio too. We will build it from scratch in this section.

### Attention, intuitively

The intuition behind attention is human. When you read a long document looking for the answer to a specific question, you do not give every sentence equal weight. You **selectively focus** on the few sentences that matter for your current question, and you mostly ignore the rest.

![Attention as selective focus](https://deeplearningwithpython.io/images/ch15/attention-concept.fde57742.png)

We want the model to do the same thing. For each word the decoder is currently producing, we want it to assign a **relevance score** to each word in the source sentence. Words with high scores contribute a lot to the next prediction; words with low scores contribute little.

![Per-token attention scores](https://deeplearningwithpython.io/images/ch15/attention.6007731a.png)

If both the source and the target are sequences (which they are, in translation), the scores form a matrix. Row *i* of the matrix tells us how much target token *i* is paying attention to each source token.

![Attention scores as a matrix](https://deeplearningwithpython.io/images/ch15/attention-scores.2932e0ff.png)

### Dot-product attention: the math

How do we actually compute these relevance scores? The simplest reasonable choice (and the one that turned out to work best after a lot of experimentation) is just the **dot product** between the target vector and each source vector. Two vectors that point in similar directions have a high dot product; two orthogonal vectors have a dot product of zero.

Once we have the raw dot-product scores, we run them through a softmax so they all add up to 1. Then we use those scores as weights and take a **weighted sum** of the source vectors. That weighted sum is the output of the attention operation. In NumPy-flavored pseudocode:

```python
scores = softmax(target @ source.T)   # shape: (target_len, source_len)
output = scores @ source              # shape: (target_len, dim)
```

That is the entire core idea of attention.

### Query, key, value: making attention learnable

To make attention richer, we do not use the raw target and source vectors directly. Instead, we project each through its own learned `Dense` layer first. The three projections are conventionally called **query**, **key**, and **value**, borrowing terminology from search engines:

- The **query** is what we project the target with. It represents "what am I currently looking for?"
- The **key** is what we project the source with. It represents "what do I, this source token, contain?"
- The **value** is *also* a projection of the source, but separate from the key. It represents "what should I contribute if I am chosen?"

Attention then becomes: take the dot product of every query with every key (these are the scores), softmax them, and use them to take a weighted sum of the values. The query and key projections control the "matching" step; the value projection controls the "what gets added in" step.

The terminology is easier to remember if you think of how an image search engine works. You type in a search query. Each photo in the database has a set of tags (keys). The engine matches your query against the keys, ranks the photos accordingly, and returns the actual photos (the values).

![Query, key, value as a search analogy](https://deeplearningwithpython.io/images/ch15/query-key-value.b57cceb0.png)

### Two refinements: scaling and multi-head

The original Transformer paper added two more refinements that turned out to matter a lot:

1. **Scaling.** When the vectors are long, dot products can get very large in magnitude. Large inputs to softmax push it into a regime where its gradient is nearly zero, and training stalls. The fix is simple: divide the scores by $\sqrt{d_k}$ (where $d_k$ is the key dimension) before applying softmax. This keeps the dot products in a reasonable range no matter how long the vectors get.
2. **Multiple heads.** Instead of doing attention once with one big query / key / value projection, we do it several times in parallel with smaller projections, then concatenate the results. Each parallel copy is called an **attention head**. Different heads can specialize: one might learn to attend to the subject of a sentence, another to the verb, another to punctuation, and so on. This avoids the "averaging blur" that happens when a single softmax has to combine too many different things at once.

![Multi-head attention](https://deeplearningwithpython.io/images/ch15/multi-head-attention.718456ad.png)

You will not need to implement any of this by hand. Keras provides `layers.MultiHeadAttention`, which does the projections, scaling, multi-head split, dot products, softmax, and reconcatenation all in one call:

```python
multi_head_attention = layers.MultiHeadAttention(num_heads=8, key_dim=64)
output = multi_head_attention(query=target, key=source, value=source)
```

### Self-attention: the move that makes the Transformer possible

Here is the move that turns attention from "a useful trick for RNNs" into "a complete replacement for RNNs." Nothing prevents us from passing the **same** sequence as the query, the key, *and* the value:

```python
output = multi_head_attention(query=source, key=source, value=source)
```

This is called **self-attention**. Every token in the sequence is allowed to look at every other token in the same sequence (including itself) and pull in information from them. Consider the sentence "The train left the station on time." When the model processes the word "station", self-attention lets it look at "train" and conclude that this is a railway station, not a radio station or a space station. The representation of "station" gets contextualized by the other words around it.

Self-attention is the building block we need to replace RNNs entirely. It can do everything an RNN does (pass information across a sequence so each token's representation depends on its neighbors), but it can do it in **one parallel step** instead of a sequential loop, and it can connect any two tokens in the sequence directly without information having to flow through every position in between.

### The encoder block

A Transformer encoder block stacks two sub-layers:

1. A **self-attention** sub-layer. This is where information flows between positions in the sequence.
2. A **feedforward** sub-layer (two `Dense` layers with a ReLU between them). This processes each position independently and adds the nonlinearity that the model needs to actually learn.

Why do we need the feedforward sub-layer at all? Because attention by itself is just a fancy linear pooling operation. Without nonlinearities, stacking attention layers would mathematically collapse to a single linear operation, and the model could not learn anything interesting. The feedforward sub-layer breaks that linearity.

Each sub-layer is wrapped with two more refinements borrowed from the ConvNet world:

- **Residual connections** (`x + sublayer(x)`). These make deep stacks trainable, just like in ResNets.
- **Layer normalization.** This is similar to batch normalization, but it normalizes across the *features* of each individual sequence rather than across the batch dimension. Layer normalization works much better than batch normalization for sequence data, where batch statistics tend to be noisy.

Finally, the layer takes a `source_mask` argument so it knows which tokens are real and which are padding. We pass this through to the attention layer's `attention_mask` argument so that padding positions are ignored when computing attention scores.

In [ ]:
class TransformerEncoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        # Self-attention sub-layer
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        # Feedforward sub-layer
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, source, source_mask):
        # --- Self-attention ---
        residual = x = source
        mask = source_mask[:, None, :]
        x = self.self_attention(query=x, key=x, value=x, attention_mask=mask)
        x = x + residual
        x = self.self_attention_layernorm(x)
        # --- Feedforward ---
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

### The decoder block

The decoder block is the encoder block plus one extra ingredient: a **cross-attention** layer that lets the decoder look at the encoder's output. So the decoder block has *three* sub-layers:

1. **Causal self-attention** over the target sequence so far. "Causal" means each position can only attend to positions that came *before* it (and itself), never to future positions. Without this restriction, the decoder could just copy the answer it is supposed to predict from the next position, and training would learn nothing useful. Keras handles this for us via `use_causal_mask=True`, which internally builds a triangular mask that zeros out future positions.
2. **Cross-attention.** The query comes from the decoder's current state, but the key and value come from the encoder's output. This is how source information flows into the decoder. It is exactly the same `MultiHeadAttention` operation we have been discussing, just with two different sequences as input.
3. The same **feedforward** sub-layer as the encoder.

Here is a visual summary of both blocks side by side:

![Encoder and decoder blocks side by side](https://deeplearningwithpython.io/images/ch15/encoder-decoder.d979dbbc.png)

In [ ]:
class TransformerDecoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        # Causal self-attention over the target
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        # Cross-attention into the encoder output
        self.cross_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.cross_attention_layernorm = layers.LayerNormalization()
        # Feedforward sub-layer
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, target, source, source_mask):
        # --- Causal self-attention over the target so far ---
        residual = x = target
        x = self.self_attention(query=x, key=x, value=x, use_causal_mask=True) # This basically sets the attention to future tokens to zero for each token.
        x = x + residual
        x = self.self_attention_layernorm(x)
        # --- Cross-attention into the source ---
        residual = x
        mask = source_mask[:, None, :]
        x = self.cross_attention(query=x, key=source, value=source, attention_mask=mask)
        x = x + residual
        x = self.cross_attention_layernorm(x)
        # --- Feedforward ---
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

**Note:** A quick note on the word "mask," because it gets reused for three different things in transformer-land.

First, the `[MASK]` token you see on the slides is a BERT-style "fill in the blank", we replace a real word with a placeholder and ask the model to predict what belongs there, using context from *both sides* of the blank. This is a training objective.

Second, in the code you'll see an `attention_mask` (sometimes called a padding mask); that's just bookkeeping to tell attention which positions are real tokens versus `[PAD]` filler added to make batches rectangular. It has nothing to do with prediction.

Third, decoder-style models (like GPT) use a causal mask, which doesn't replace any tokens but instead prevents each position from attending to positions to its right, so the model can only use past context to predict the next word. Same English word, three unrelated mechanisms; don't let the shared terminology trip you up.

### Positional embeddings: making the Transformer order-aware

There is a subtle but huge problem with what we have built so far. **Attention is order-blind.** If you shuffle the words of an input sentence, the set of pairwise dot products is exactly the same (just permuted), so the output of every attention layer is exactly the same (just permuted). This means a Transformer with no extra ingredients would treat "dog bites man" and "man bites dog" identically. Clearly, that is not what we want for a model that is supposed to understand language.

The fix is to **inject position information directly into the embeddings**. The simplest version, and the one we will use, is this: give each position in the sequence its own learned embedding vector, and *add* it elementwise to the word embedding at that position. Position 0 gets one vector, position 1 gets another, and so on, all learned from scratch alongside the rest of the model. After this addition, the input to the attention layer encodes both *what* the token is (from the word embedding) and *where* it appears (from the position embedding).

In [ ]:
from keras import ops

class PositionalEmbedding(keras.Layer):
    def __init__(self, sequence_length, input_dim, output_dim):
        super().__init__()
        self.token_embeddings = layers.Embedding(input_dim, output_dim)
        self.position_embeddings = layers.Embedding(sequence_length, output_dim)

    def call(self, inputs):
        positions = ops.cumsum(ops.ones_like(inputs), axis=-1) - 1
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

### Putting it all together: a Transformer translator

Now we have every piece we need. The full model:

1. Embed the English source with `PositionalEmbedding`.
2. Pass the result through a `TransformerEncoder`, which produces a contextualized representation of every English token.
3. Embed the Spanish target (so far) with `PositionalEmbedding`.
4. Pass the embedded target plus the encoder output through a `TransformerDecoder`.
5. Project the decoder output to a softmax over the Spanish vocabulary, producing a probability distribution over the next Spanish word at each position.

In [ ]:
hidden_dim = 256
intermediate_dim = 2048
num_heads = 8

source = keras.Input(shape=(None,), dtype="int32", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(source)
encoder_output = TransformerEncoder(hidden_dim, intermediate_dim, num_heads)(
    source=x,
    source_mask=source != 0,
)

target = keras.Input(shape=(None,), dtype="int32", name="spanish")
x = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)(target)
x = TransformerDecoder(hidden_dim, intermediate_dim, num_heads)(
    target=x,
    source=encoder_output,
    source_mask=source != 0,
)
x = layers.Dropout(0.5)(x)
target_predictions = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([source, target], target_predictions)
transformer.summary(line_length=80)

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ english (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ positional_embedding  │ (None, None, 256) │   3,845,120 │ english[0][0]      │
│ (PositionalEmbedding) │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal_3           │ (None, None)      │           0 │ english[0][0]      │
│ (NotEqual)            │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ spanish (InputLayer)  │ (None, None)      │           0 │ -                  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_encoder   │ (None, None, 256) │   1,315,072 │ positional_embedd… │
│ (TransformerEncoder)  │                   │             │ not_equal_3[0][0]  │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ not_equal_4           │ (None, None)      │           0 │ english[0][0]      │
│ (NotEqual)            │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ positional_embedding… │ (None, None, 256) │   3,845,120 │ spanish[0][0]      │
│ (PositionalEmbedding) │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_decoder   │ (None, None, 256) │   1,578,752 │ transformer_encod… │
│ (TransformerDecoder)  │                   │             │ not_equal_4[0][0], │
│                       │                   │             │ positional_embedd… │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dropout_6 (Dropout)   │ (None, None, 256) │           0 │ transformer_decod… │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ dense_8 (Dense)       │ (None, None,      │   3,855,000 │ dropout_6[0][0]    │
│                       │ 15000)            │             │                    │
└───────────────────────┴───────────────────┴─────────────┴────────────────────┘

 Total params: 14,439,064 (55.08 MB)

 Trainable params: 14,439,064 (55.08 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
transformer.fit(train_ds, epochs=30, validation_data=val_ds)

Epoch 1/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 46s 30ms/step - accuracy: 0.3991 - loss: 1.4028 - val_accuracy: 0.5582 - val_loss: 0.9134
Epoch 2/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 29s 22ms/step - accuracy: 0.5953 - loss: 0.8467 - val_accuracy: 0.6340 - val_loss: 0.7105
Epoch 3/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 29s 22ms/step - accuracy: 0.6641 - loss: 0.6449 - val_accuracy: 0.6631 - val_loss: 0.6367
Epoch 4/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 29s 22ms/step - accuracy: 0.7038 - loss: 0.5307 - val_accuracy: 0.6755 - val_loss: 0.6145
Epoch 5/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 29s 22ms/step - accuracy: 0.7322 - loss: 0.4559 - val_accuracy: 0.6822 - val_loss: 0.6057
Epoch 6/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 29s 22ms/step - accuracy: 0.7550 - loss: 0.4005 - val_accuracy: 0.6851 - val_loss: 0.6068
Epoch 7/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 29s 22ms/step - accuracy: 0.7735 - loss: 0.3592 - val_accuracy: 0.6878 - val_loss: 0.6070
Epoch 8/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 29s 22ms/step - accuracy: 0.7893 -

In [ ]:
import numpy as np

spa_vocab = spanish_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence = english_tokenizer([input_sentence])
    decoded_sentence = "[start]"
    for i in range(sequence_length):
        tokenized_target_sentence = spanish_tokenizer([decoded_sentence])
        tokenized_target_sentence = tokenized_target_sentence[:, :-1]
        inputs = [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions = transformer.predict(inputs, verbose=0)
        sampled_token_index = np.argmax(next_token_predictions[0, i, :])
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

-
It so happened that I had no money with me.
[start] eso que me pasó por algo de dinero [end]
-
Where does that idiot think he is going?
[start] dónde está pasando eso en ir [end]
-
Tom took the stairs two at a time.
[start] tom tomó las dos de la razón [end]
-
What are you talking about?
[start] de qué estás hablando [end]
-
I understand why.
[start] entiendo el porqué [end]


You should reach roughly **67% next-token accuracy**, which is better than the GRU baseline (65%), even though the Transformer has about **half as many parameters**. Two more things are worth noticing:

- **Each epoch is much faster than an RNN epoch**, even at the same parameter count. There is no sequential loop during training; attention computes the whole sequence in one big parallel matrix multiplication. This parallelism is the main reason Transformers scale so gracefully to large datasets and large hardware.
- The model is still improving at epoch 30. Transformers are **data-hungry**: give them more data and they keep getting better, while RNNs plateau much earlier.

If you want to actually generate Spanish translations from this model, the procedure is exactly the same autoregressive loop we discussed for Shakespeare, just with one extra input: feed the English source into the encoder once, then start the decoder with the `[start]` token, sample one Spanish token at a time using argmax over the decoder's output, and stop when the model emits `[end]`. The book provides a `generate_translation` function in Chapter 15 that implements this in a few lines. Subjectively, the trained Transformer produces noticeably better translations than the GRU baseline, though both are still toy models.

## Classification with a pretrained Transformer (RoBERTa on IMDb)

So far we have built a Transformer from scratch and watched it translate. But the real superpower of the Transformer is how well it **scales**. The more data and the more parameters you throw at it, the better it gets. This led directly to the era of **pretrained foundation models**: train one giant Transformer once, on hundreds of gigabytes of text, then fine-tune it on whatever your downstream task is.

The first such model to become famous was **BERT**, released in 2018. Shortly after came **RoBERTa** (2019), a refined and more heavily trained version of BERT. RoBERTa was trained on 160 GB of internet text. The "base" version we will use has **124 million parameters** (almost ten times bigger than the Transformer we just built), and it has already learned a rich, general-purpose representation of English from its pretraining.

How was it pretrained? Both BERT and RoBERTa use **masked language modeling**, which is a slightly different setup from the causal language modeling we did for Shakespeare. Instead of asking the model to predict the next token from past tokens only, masked language modeling randomly hides about 15% of the tokens in a sentence (replacing them with a special `[MASK]` token) and asks the model to fill in the blanks using both the *left* and the *right* context. This is unsupervised: any plain text works, no labels needed, which is why it scales so easily.

Now we will load RoBERTa and fine-tune it on the IMDb sentiment task we struggled with at the end of Part I. The hope: a model that already "knows English" should need very little task-specific training to figure out which patterns mean "positive review" and which mean negative review.

In [ ]:
import keras_hub

tokenizer = keras_hub.models.Tokenizer.from_preset("roberta_base_en")
backbone = keras_hub.models.Backbone.from_preset("roberta_base_en")

In [ ]:
tokenizer("The quick brown fox")

Array([  133,  2119,  6219, 23602], dtype=int32)

Notice that RoBERTa's tokenizer turns those four English words into just four token IDs. That is RoBERTa's BPE-style subword tokenizer at work. (Yes, the same BPE algorithm built by hand in Chapter 14!)

The `Backbone` is the entire pretrained Transformer encoder: 12 stacked `TransformerEncoder` layers, each one structurally identical to the encoder block we wrote earlier in this notebook, just much bigger:

In [ ]:
backbone.summary(line_length=80)

Model: "roberta_backbone"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ token_ids             │ (None, None)      │           0 │ -                  │
│ (InputLayer)          │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embeddings            │ (None, None, 768) │  38,996,736 │ token_ids[0][0]    │
│ (TokenAndPositionEmb… │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embeddings_layer_norm │ (None, None, 768) │       1,536 │ embeddings[0][0]   │
│ (LayerNormalization)  │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ embeddings_dropout    │ (None, None, 768) │           0 │ embeddings_layer_… │
│ (Dropout)             │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ padding_mask          │ (None, None)      │           0 │ -                  │
│ (InputLayer)          │                   │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_0   │ (None, None, 768) │   7,087,872 │ embeddings_dropou… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_1   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_2   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_3   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_4   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_5   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_6   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_7   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_8   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_9   │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │                   │             │ padding_mask[0][0] │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ transformer_layer_10  │ (None, None, 768) │   7,087,872 │ transformer_layer… │
│ (TransformerEncoder)  │      

 Total params: 124,052,736 (473.22 MB)

 Trainable params: 124,052,736 (473.22 MB)

 Non-trainable params: 0 (0.00 B)

### Preprocessing IMDb for RoBERTa

When fine-tuning a pretrained model, you must use **its exact tokenizer and its exact input format**. The model's learned weights are tied to that specific vocabulary. If you pass tokens it does not recognize, the embeddings will be meaningless.

RoBERTa expects each input to start with a `<s>` token, end with a `</s>` token, and be padded with `<pad>` tokens. KerasHub's `StartEndPacker` layer handles all of this for us in one call.

In [ ]:
def preprocess(text, label):
    packer = keras_hub.layers.StartEndPacker(
        sequence_length=512,
        start_value=tokenizer.start_token_id,
        end_value=tokenizer.end_token_id,
        pad_value=tokenizer.pad_token_id,
        return_padding_mask=True,
    )
    token_ids, padding_mask = packer(tokenizer(text))
    return {"token_ids": token_ids, "padding_mask": padding_mask}, label

# We can reuse the train_dir / val_dir / test_dir from Part I.
batch_size = 16
train_ds = text_dataset_from_directory(train_dir, batch_size=batch_size)
val_ds   = text_dataset_from_directory(val_dir,   batch_size=batch_size)
test_ds  = text_dataset_from_directory(test_dir,  batch_size=batch_size)

preprocessed_train_ds = train_ds.map(preprocess)
preprocessed_val_ds   = val_ds.map(preprocess)
preprocessed_test_ds  = test_ds.map(preprocess)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


### Adding a classification head and fine-tuning

The pretrained backbone outputs a sequence of 768-dimensional vectors, one per input token. To make a single yes/no classification decision per review, we need to collapse that sequence into a single vector. The standard trick is to use the **first token's** vector (`x[:, 0, :]`).

Why the first token? Because attention is global. By the time we reach the final encoder layer, the very first token's representation has had a chance to attend to every other token in the sequence and pull in information from all of them. So the first token's output vector ends up being a contextualized summary of the entire review.

We then pass that single vector through a small classification head: dropout, a `Dense` layer with ReLU, more dropout, and a final sigmoid output for the binary positive/negative decision.

In [ ]:
inputs = backbone.input
x = backbone(inputs)
x = x[:, 0, :]
x = layers.Dropout(0.1)(x)
x = layers.Dense(768, activation="relu")(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
classifier = keras.Model(inputs, outputs)

Two important things to know about training a fine-tuning run.

1. **Use a very small learning rate** (5e-5 here). Fine-tuning means making tiny adjustments to weights that are already good. A large learning rate would destroy everything the model learned during pretraining.
2. **One epoch is usually enough.** The model already knows English from its pretraining. We just need to nudge it to apply that knowledge to the binary positive/negative review classification task.

In [ ]:
classifier.compile(
    optimizer=keras.optimizers.Adam(5e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
classifier.fit(
    preprocessed_train_ds,
    validation_data=preprocessed_val_ds,
    epochs=1,
)
classifier.evaluate(preprocessed_test_ds)

1250/1250 ━━━━━━━━━━━━━━━━━━━━ 2154s 2s/step - accuracy: 0.9047 - loss: 0.2543 - val_accuracy: 0.9260 - val_loss: 0.1879
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 821s 522ms/step - accuracy: 0.9225 - loss: 0.1895


[0.18950806558132172, 0.9225199818611145]

You should hit somewhere around **93% test accuracy** after just one epoch of fine-tuning. Let that sink in. In Part I, our best model topped out at 90% with bigrams, and the 2-million-parameter LSTM only managed 84%. Now a single epoch of fine-tuning a pretrained model has cleared 93%. The larger RoBERTa-large checkpoint can push past 95%.

This is the modern NLP recipe in a nutshell: **do not train your big models from scratch on small datasets. Pretrain (or download a pretrained model) on huge amounts of unlabeled text, then fine-tune on your small labeled dataset.**

## What makes the Transformer effective?

Let's step back. *Why* does this architecture work so well?

The honest answer is that the field discovered most of it empirically. People tried things, kept what worked, and the Transformer is what survived. But there is a clean conceptual story you can tell about it.

A Transformer is fundamentally a machine for **learning structured embedding spaces**. Each layer of the model takes a sequence of token vectors and produces a new sequence of token vectors, where each new vector is a *weighted, context-dependent recombination* of the old ones. (That recombination is exactly what the attention operation does.) Tokens that frequently appear together, and tokens whose vectors already point in similar directions, get pulled even closer together with each successive layer. Over many layers, this process produces a rich semantic space where:

- Distance encodes meaning: nearby vectors mean similar things, far-apart vectors mean different things.
- The space is **continuous and interpolative**. You can move smoothly from one meaning to another, and intermediate points represent intermediate meanings.
- It can store enormously complex patterns. Not just simple "king minus man plus woman equals queen" arithmetic from the old Word2Vec embeddings, but entire **vector programs** like "rewrite this paragraph in the style of Shakespeare."

At the same time, two practical properties make Transformers scale where other architectures do not:

1. **Parallelism.** No sequential loops at training time. Attention is one big matrix multiply, which maps perfectly onto modern GPUs and TPUs.
2. **Data appetite.** Unlike RNNs, Transformers do not saturate quickly. Give them more data and more parameters, and they keep getting better. This is what made models like GPT, BERT, and Llama possible.

It is also worth keeping a sober view. These models are extraordinarily good at **interpolating** within the patterns they have seen, but they do not "understand" language the way humans do, and they will confidently make things up (a phenomenon called *hallucination*) when pushed beyond their training distribution. The Transformer is a beautiful, powerful pattern-recognition machine. It is not a mind.

## Summary

**Chapter 14: Text classification**

- Text must be **tokenized** before any model can use it. The standard pipeline has three stages: standardize, split, and index.
- There are three families of tokenizers: **character**, **word**, and **subword** (BPE). Subword is the modern default and powers GPT, BERT, RoBERTa, and friends.
- For text classification, you can treat text as an unordered **set** of tokens (bag-of-words, bigrams) or as an ordered **sequence**.
- Set models with bigrams give you about 90% IMDb test accuracy with only 30K parameters. They are astonishingly efficient.
- Sequence models need much more data to pay off, which is why pretraining matters so much.
- A **word embedding** is a learned dense vector per token. It is vastly more efficient and meaningful than one-hot encoding.

**Chapter 15: Language models and the Transformer**

- A **language model** predicts $p(\text{next token} \mid \text{past tokens})$ and can be sampled in a loop to generate arbitrarily long text.
- **Sequence-to-sequence** problems (like translation) use an **encoder-decoder** setup, where the encoder summarizes the source and the decoder generates the target one token at a time.
- **Attention** lets every token in a sequence selectively pull information from any other token. This breaks the long-range memory bottleneck of RNNs.
- The key formula: scores = softmax(query times key transpose, divided by square root of d), output = scores times value. **Multi-head attention** runs several of these in parallel with different projections.
- A **Transformer encoder block** combines self-attention, a feedforward sub-layer, residual connections, and layer normalization. A **decoder block** adds *causal* self-attention and a *cross-attention* into the encoder output.
- Attention is order-blind, so we add **positional embeddings** to the token embeddings.
- Transformers train in parallel (no sequential loops) and scale gracefully with data and parameters.
- Modern NLP equals **pretrain on huge corpora, then fine-tune** on your task. Fine-tuning RoBERTa for one epoch beats every from-scratch model in this notebook.